# Adama — Tor EC vs. FTIR-EC (ground-truth cross-plot)

**Goal (2026-06-25 meeting, Adama step 2).** For the five Adama samples with thermal-optical
reflectance (**Tor**) carbon, plot **Tor EC on x** (the ground truth) against **FTIR-EC on y**, with
the **1:1 line**. Points **below 1:1** are where **FTIR under-reports EC vs. Tor** — the central
hypothesis. Do this for the **general** calibration now; overlay the **biomass** calibration when its
predictions land.

> The advisors' correction: *don't* plot FTIR-vs-FTIR. Tor is the reference we are trying to
> reproduce, so everything is measured against it.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

CANDS = [Path("../spartan_ec_2026_06_16/data/adama"),
         Path("../../spartan_ec_2026_06_16/data/adama"),
         Path("spartan_ec_2026_06_16/data/adama")]
ADAMA = next((p for p in CANDS if (p / "adama_quartz_tor_batch54.csv").exists()), None)
assert ADAMA is not None, "adama_quartz_tor_batch54.csv not found in: " + ", ".join(map(str, CANDS))
print("Adama data dir:", ADAMA)
Path("figures").mkdir(exist_ok=True); Path("tables").mkdir(exist_ok=True)

Adama data dir: ../spartan_ec_2026_06_16/data/adama


## Tor EC per quartz filter

`ECTR` = EC by **reflectance** (the Tor convention) = `EC1+EC2+EC3 − OPTR`. We take the file's
`Concentration_ug_m3` (ambient concentration) so it is directly comparable to the FTIR-EC
concentration, even though the two measurements are on different substrates (quartz vs. Teflon).

In [2]:
tor = pd.read_csv(ADAMA / "adama_quartz_tor_batch54.csv")
tor["date"] = pd.to_datetime(tor["SampleDate"]).dt.normalize()
tor_ec = (tor[tor["Parameter"] == "ECTR"]
          .set_index("FilterId")[["date", "Concentration_ug_m3"]]
          .rename(columns={"Concentration_ug_m3": "Tor_EC_ug_m3"}))
tor_ec.index.name = "quartz_FilterId"
print("Tor EC (reflectance) per quartz filter:")
print(tor_ec.to_string())

Tor EC (reflectance) per quartz filter:
                      date  Tor_EC_ug_m3
quartz_FilterId                         
J1679           2024-07-30      3.040817
J1693           2024-07-09      3.378748
J1703           2024-07-26      1.861959
J1701           2024-07-20      2.397885
J1675           2024-07-23      2.449553


## General FTIR-EC per PTFE filter, joined to Tor by date

In [3]:
ft = pd.read_csv(ADAMA / "adama_ptfe_ftir_batch54.csv")
ft["date"] = pd.to_datetime(ft["SampleDate"]).dt.normalize()
ftir_ec = (ft[ft["Parameter"] == "EC_ftir"]
           .set_index("FilterId")[["date", "Concentration_ug_m3"]]
           .rename(columns={"Concentration_ug_m3": "FTIR_EC_general_ug_m3"}))
ftir_ec.index.name = "ptfe_FilterId"

pair = (tor_ec.reset_index().merge(ftir_ec.reset_index(), on="date", how="inner")
        .sort_values("date").reset_index(drop=True))
assert len(pair) == 5, f"expected 5 co-located pairs, got {len(pair)}"
print("Paired quartz(Tor) <-> PTFE(FTIR) by sample date:")
print(pair.to_string(index=False))

Paired quartz(Tor) <-> PTFE(FTIR) by sample date:
quartz_FilterId       date  Tor_EC_ug_m3 ptfe_FilterId  FTIR_EC_general_ug_m3
          J1693 2024-07-09      3.378748         J1269               5.276551
          J1701 2024-07-20      2.397885         J1266               1.532098
          J1675 2024-07-23      2.449553         J1285               1.991587
          J1703 2024-07-26      1.861959         J1233               1.293041
          J1679 2024-07-30      3.040817         J1270               1.766609


## INPUT — biomass-calibration FTIR-EC (paste when available)

Fill in the biomass-only calibration's EC prediction (µg/m³) for each **PTFE FilterId**. Leave as
`None` to skip the biomass overlay. Once `05_calibration_variants` produces these, paste them here
and re-run — the plot and table update automatically.

In [4]:
BIOMASS_EC_ug_m3 = {
    "J1233": None,
    "J1266": None,
    "J1269": None,
    "J1270": None,
    "J1285": None,
}
pair["FTIR_EC_biomass_ug_m3"] = pair["ptfe_FilterId"].map(BIOMASS_EC_ug_m3)
has_biomass = pair["FTIR_EC_biomass_ug_m3"].notna().any()
print("biomass overlay:", "ON" if has_biomass else "OFF (placeholders still None)")
pair

biomass overlay: OFF (placeholders still None)


,quartz_FilterId,date,Tor_EC_ug_m3,ptfe_FilterId,FTIR_EC_general_ug_m3,FTIR_EC_biomass_ug_m3
0,J1693,2024-07-09,3.378748,J1269,5.276551,None
1,J1701,2024-07-20,2.397885,J1266,1.532098,None
2,J1675,2024-07-23,2.449553,J1285,1.991587,None
3,J1703,2024-07-26,1.861959,J1233,1.293041,None
4,J1679,2024-07-30,3.040817,J1270,1.766609,None


## Cross-plot — Tor EC (x) vs. FTIR-EC (y), 1:1 reference

In [5]:
fig, ax = plt.subplots(figsize=(6.6, 6.2))
lim = float(np.nanmax([pair["Tor_EC_ug_m3"].max(),
                       pair["FTIR_EC_general_ug_m3"].max(),
                       (pair["FTIR_EC_biomass_ug_m3"].max() if has_biomass else 0)])) * 1.15
ax.plot([0, lim], [0, lim], "k--", lw=1, label="1:1 (FTIR = Tor)")

ax.scatter(pair["Tor_EC_ug_m3"], pair["FTIR_EC_general_ug_m3"],
           s=90, color="#1f77b4", zorder=3, label="general FTIR-EC")
if has_biomass:
    ax.scatter(pair["Tor_EC_ug_m3"], pair["FTIR_EC_biomass_ug_m3"],
               s=90, color="#d62728", marker="^", zorder=3, label="biomass FTIR-EC")

for _, r in pair.iterrows():
    ax.annotate(f'{r["ptfe_FilterId"]}/{r["quartz_FilterId"]}',
                (r["Tor_EC_ug_m3"], r["FTIR_EC_general_ug_m3"]),
                textcoords="offset points", xytext=(7, 3), fontsize=8)

ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel("Tor EC (reflectance, µg/m³)  — ground truth")
ax.set_ylabel("FTIR-EC (µg/m³)")
ax.set_title("Adama — Tor EC vs. FTIR-EC (n=5)")
ax.text(lim*0.97, lim*0.03, "below 1:1 → FTIR under-reports", ha="right", va="bottom",
        fontsize=8, color="gray", style="italic")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.savefig("figures/fig02_adama_tor_vs_ftir.png", dpi=140, bbox_inches="tight")
print("saved figures/fig02_adama_tor_vs_ftir.png"); plt.show()

saved figures/fig02_adama_tor_vs_ftir.png


/var/folders/7k/65ckdzsj0w3171qv80rh8dgr0000gn/T/com.apple.shortcuts.mac-helper/ipykernel_81504/2560183848.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  print("saved figures/fig02_adama_tor_vs_ftir.png"); plt.show()


## The numbers — ratio to Tor, and where each sample sits

In [6]:
out = pair.copy()
out["general/Tor"] = out["FTIR_EC_general_ug_m3"] / out["Tor_EC_ug_m3"]
if has_biomass:
    out["biomass/Tor"] = out["FTIR_EC_biomass_ug_m3"] / out["Tor_EC_ug_m3"]
out["side_of_1:1(general)"] = np.where(out["general/Tor"] < 1, "below (FTIR<Tor)", "above (FTIR>Tor)")
cols = ["date", "quartz_FilterId", "ptfe_FilterId", "Tor_EC_ug_m3",
        "FTIR_EC_general_ug_m3", "general/Tor", "side_of_1:1(general)"]
print(out[cols].round(3).to_string(index=False))
out.round(4).to_csv("tables/adama_tor_vs_ftir.csv", index=False)
print("\nwrote tables/adama_tor_vs_ftir.csv")
print("\nmedian general/Tor ratio:", round(float(out['general/Tor'].median()), 3))

      date quartz_FilterId ptfe_FilterId  Tor_EC_ug_m3  FTIR_EC_general_ug_m3  general/Tor side_of_1:1(general)
2024-07-09           J1693         J1269         3.379                  5.277        1.562     above (FTIR>Tor)
2024-07-20           J1701         J1266         2.398                  1.532        0.639     below (FTIR<Tor)
2024-07-23           J1675         J1285         2.450                  1.992        0.813     below (FTIR<Tor)
2024-07-26           J1703         J1233         1.862                  1.293        0.694     below (FTIR<Tor)
2024-07-30           J1679         J1270         3.041                  1.767        0.581     below (FTIR<Tor)

wrote tables/adama_tor_vs_ftir.csv

median general/Tor ratio: 0.694


### Reading it next week
- Points **below the 1:1 line** are the "FTIR misses EC" cases — call them out by FilterId.
- The **J1269/J1693** pair (2024-07-09) is the anomalous high-EC day flagged in the meeting — see
  where it lands and whether the biomass calibration moves it *toward* Tor (expected) or the wrong
  way (the meeting noted one sample moves opposite).
- Pair this with `03_adama_char_soot`: samples that are **char-rich** should be the ones FTIR
  under-reports most. That linkage is the story.